# RightSize AI — benchmark run

Everything runs here. The outputs are two static files, `results.json` and `replay.json`, in `/kaggle/working/`.

**Before "Save & Run All"**
1. Settings → **Accelerator: GPU T4 x2** · **Internet: on**
2. Add-ons → Secrets: `MISTRAL_API_KEY`, `HF_TOKEN` (and `ANTHROPIC_API_KEY` for a full run) (accept the MILU terms on huggingface.co first)
3. **Resuming?** Add the previous version's output as input (Add Input → Your Work → this notebook). The DB is picked up automatically and finished calls are skipped.

`QUICK = True` (the default, set in the first code cell) runs 7 configs × 30 items per task in about 1–2 hours: real measurements, wider confidence intervals. Set it to `False` for the full run.

A full run is longer than one 12-hour session. The harness stops cleanly before the limit; save the version, attach its output, and run again.

In [ ]:
REPO = "https://github.com/Himanshusolanki24/Global-.git"
BRANCH = "main"
WORK = "/kaggle/working"
DB = f"{WORK}/rightsize.db"
TASKS = f"{WORK}/tasks"       # kept with the output, so a resumed run uses the identical items
CODE = "/tmp/rightsize"       # code is not output
BUDGET_H = 11.0               # whole-notebook budget; Kaggle stops sessions at 12 h
USE_MISTRAL = False   # True once your Mistral key answers (console.mistral.ai → Limits shows an active plan)
SMOKE = ["qwen2.5-0.5b/Q4"] + (["mistral-small/API"] if USE_MISTRAL else [])

# QUICK = True: a real but smaller run that fits one session (~1–2 h). Every number is measured;
# the CIs are just wider (30 items per task). Set QUICK = False for the full 17-config, 200-item run.
QUICK = True
QUICK_CONFIGS = [
    "qwen2.5-0.5b/Q4", "llama3.2-1b/Q4", "gemma2-2b/Q4",   # the size ladder
    "phi3.5-mini/Q4", "phi3.5-mini/FP16",                  # router small model + its quantization pair
    "llama3.1-8b/Q8",                                      # router big model (local, energy measured)
] + (["mistral-small/API"] if USE_MISTRAL else [])           # cloud big model
RUN_ARGS = f"--limit 30 --energy-items 10 --configs {' '.join(QUICK_CONFIGS)}" if QUICK else ""

import glob, os, shutil, subprocess, sys, time, urllib.request
START = time.time()

def hours_left() -> float:
    return round(BUDGET_H - (time.time() - START) / 3600, 2)

def sh(cmd: str) -> None:
    """Run a command, streaming its output into the cell; raise if it fails."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=CODE if os.path.isdir(CODE) else None)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for k in ["MISTRAL_API_KEY", "HF_TOKEN", "ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
        try:
            os.environ[k] = secrets.get_secret(k)
        except Exception:
            print(f"secret {k} is not set")
except ImportError:
    print("not on Kaggle: using the existing environment")

## 1 · Ollama, installed natively, one model loaded at a time

In [ ]:
sh("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null")
sh("curl -fsSL https://ollama.com/install.sh | sh")
env = {**os.environ, "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_KEEP_ALIVE": "30m"}
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{WORK}/ollama.log", "a"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("ollama", urllib.request.urlopen("http://localhost:11434/api/version").read().decode())
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("ollama serve did not start — see /kaggle/working/ollama.log")
sh("nvidia-smi --query-gpu=index,name,memory.total,power.draw --format=csv")

## 2 · Code, packages, and resume state

In [ ]:
if not os.path.isdir(CODE):
    sh(f"git clone --depth 1 -b {BRANCH} {REPO} {CODE}")
else:
    sh(f"git -C {CODE} pull --ff-only")  # pick up fixes pushed since this session started
sh("pip install -q -r requirements.txt")

for src in sorted(glob.glob("/kaggle/input/*/rightsize.db"))[-1:]:
    if not os.path.exists(DB):
        shutil.copy(src, DB)
        print("resuming from", src)
for src in sorted(glob.glob("/kaggle/input/*/tasks"))[-1:]:
    if not os.path.isdir(TASKS):
        shutil.copytree(src, TASKS)
        print("reusing task bank from", src)

## 3 · Task bank (seeded; built once, then reused) and model tags (checked before any download)

In [ ]:
if len(glob.glob(f"{TASKS}/*.jsonl")) < 6:
    sh(f"python -m bench.data.build_tasks --out {TASKS}")
sh("python -c \"from bench.core.schemas import load_configs; from bench.runners.ollama_runner import verify_tags; "
   "t=[c.tag for c in load_configs() if c.backend=='ollama']; verify_tags(t); print(len(t), 'ollama tags found on the registry')\"")

## 4 · Host fingerprint and 60 s idle baseline (no model loaded)

In [ ]:
sh(f"python -m bench.core.host --db {DB} --idle-seconds 60")

## 5 · Smoke run: 3 items × 1 local + 1 API model. Its rows are real and are reused by the full run.

In [ ]:
sh(f"python -m bench.harness --db {DB} --tasks-dir {TASKS} --limit 3 --strict --configs {' '.join(SMOKE)}")
import sqlite3
errs = sqlite3.connect(DB).execute("SELECT config_id, task_id, error FROM runs WHERE error IS NOT NULL").fetchall()
if errs:
    for e in errs[:20]:
        print(e)
    raise RuntimeError(f"smoke run has {len(errs)} errored calls — fix before the full run")
print("smoke run clean")

## 6 · Full benchmark (resumable; stops cleanly when the session budget runs out)

In [ ]:
sh(f"python -m bench.harness --db {DB} --tasks-dir {TASKS} {RUN_ARGS} --deadline-hours {hours_left() - 1.0}")

## 7 · Router confidence samples (small model, temperature 0.7, 3 samples, val + test items)

In [ ]:
sh(f"python -m bench.harness --db {DB} --tasks-dir {TASKS} --phase conf {RUN_ARGS} --deadline-hours {hours_left() - 0.4}")

## 8 · Train the router, then export

In [ ]:
sh(f"python -m bench.router.train --db {DB} --tasks-dir {TASKS}")
sh(f"python -m bench.export --db {DB} --out {WORK}")

In [ ]:
import json
r = json.load(open(f"{WORK}/results.json"))
print(f"{len(r['results'])} cells · {len(r['configs'])} configs · run {r['run']['id']} on {r['run']['hardware']}")
for v in r["router"]["variants"]:
    s = v["sim"]
    print(f"router → {v['big']}: τ={v['tau']:.2f}  quality kept {s['quality_retained_pct']}%  "
          f"₹ saved/1k {s['inr_saved_1k']}  Wh saved/1k {s['wh_saved_1k'] if s['wh_saved_1k'] is not None else 'not measurable'}")
print("Download results.json + replay.json from the Output tab → commit to frontend/public/")